## Train projection on top of projection in clip

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset, Subset
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from pytorch_metric_learning.losses import CircleLoss
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 32
learning_rate = 1e-3
num_epochs = 50

# Directories
train_dir = '' # reid train dir
test_dir = '' # reid test dir
gallery_test_dir = '' # reid gallery_test dir

def extract_label(filename):
    # Label is first 4 digits of the filename
    return int(filename[:4])

class EmbeddingDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        for file in os.listdir(root_dir):
            if file.endswith('.pt'):
                label = int(file[:4])
                path = os.path.join(root_dir, file)
                self.samples.append((path, label))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        embedding = torch.load(path)
        if embedding.dim() > 1:
            embedding = embedding.squeeze()
        if self.transform:
            embedding = self.transform(embedding)
        return embedding, label, idx

# Datasets and loaders
train_dataset = EmbeddingDataset(train_dir)
test_dataset = EmbeddingDataset(test_dir)
gallery_test_dataset = EmbeddingDataset(gallery_test_dir)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
gallery_test_loader = DataLoader(gallery_test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

class ProjectionModel(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=512, output_dim=768):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
            nn.BatchNorm1d(output_dim)
        )
    def forward(self, x):
        return self.net(x)

model = ProjectionModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = CircleLoss()

def get_embeddings(model, loader, target_classes=None, max_samples_per_class=50, device=device, is_gallery=False, return_idxs=False):
    model.eval()
    embeddings, labels, idxs, is_gallery_list = [], [], [], []
    with torch.no_grad():
        pbar = tqdm(total=len(loader.dataset), desc="Extracting embeddings")
        if target_classes is not None and max_samples_per_class is not None:
            dataset = loader.dataset
            samples = dataset.samples
            selected_indices = set()
            for cls in target_classes:
                cls_indices = [idx for idx, (_, label) in enumerate(samples) if label == cls]
                if len(cls_indices) > max_samples_per_class:
                    random.shuffle(cls_indices)
                    cls_indices = cls_indices[:max_samples_per_class]
                selected_indices.update(cls_indices)
            subset = Subset(dataset, list(selected_indices))
            loader = DataLoader(subset, batch_size=batch_size, shuffle=False, num_workers=4)
        for batch in loader:
            emb_batch, labels_batch, idxs_batch = batch
            emb_batch = emb_batch.to(device)
            out = model(emb_batch)
            out = out.cpu().numpy()
            embeddings.append(out)
            labels.append(labels_batch.numpy())
            idxs.append(idxs_batch.numpy())
            is_gallery_list.append([is_gallery] * emb_batch.size(0))
            pbar.update(emb_batch.size(0))
        pbar.close()
    embeddings = np.concatenate(embeddings)
    labels = np.concatenate(labels)
    is_gallery = np.concatenate(is_gallery_list)
    if return_idxs:
        idxs = np.concatenate(idxs)
        return embeddings, labels, is_gallery, idxs
    return embeddings, labels, is_gallery

def plot_clusters(embeddings, labels, target_classes=None, epoch=0, is_gallery=None):
    tsne = TSNE(n_components=2, random_state=42)
    embeddings_2d = tsne.fit_transform(embeddings)
    plt.figure(figsize=(10,8))
    if is_gallery is not None:
        # Plot query points
        query_mask = ~is_gallery
        query_points = embeddings_2d[query_mask]
        query_labels = labels[query_mask]
        unique_labels = np.unique(query_labels) if target_classes is None else target_classes
        for label in unique_labels:
            mask = query_labels == label
            points = query_points[mask]
            plt.scatter(points[:,0], points[:,1], marker='o', s=100,
                        label=f'Class {label} (Query)')
        # Plot gallery points
        gallery_mask = is_gallery
        gallery_points = embeddings_2d[gallery_mask]
        gallery_labels = labels[gallery_mask]
        for label in unique_labels:
            mask = gallery_labels == label
            points = gallery_points[mask]
            plt.scatter(points[:,0], points[:,1], marker='*', s=200,
                        label=f'Class {label} (Gallery)')
    else:
        unique_labels = np.unique(labels) if target_classes is None else target_classes
        for label in unique_labels:
            mask = labels == label
            points = embeddings_2d[mask]
            plt.scatter(points[:,0], points[:,1], label=f'Class {label}')
    plt.title(f'Epoch {epoch} Clusters (t-SNE)')
    plt.xlabel('t-SNE Dimension 1')
    plt.ylabel('t-SNE Dimension 2')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

def evaluate(model, gallery_loader, query_loader, target_classes, device, epoch=0, log_file='evaluation.log'):
    model.eval()
    gallery_emb, gallery_labels, _ = get_embeddings(model, gallery_loader, target_classes, device=device)
    query_emb, query_labels, _ = get_embeddings(model, query_loader, target_classes, device=device)
    gallery_emb = F.normalize(torch.from_numpy(gallery_emb).to(device), p=2, dim=1)
    query_emb = F.normalize(torch.from_numpy(query_emb).to(device), p=2, dim=1)
    similarity_matrix = torch.mm(query_emb, gallery_emb.T)
    num_query = len(query_labels)
    num_gallery = len(gallery_labels)
    ap_list, rank1, rank3, rank5 = [], 0, 0, 0
    gallery_labels = np.array(gallery_labels)
    query_labels = np.array(query_labels)
    for i in range(num_query):
        sim_scores = similarity_matrix[i]
        true_label = query_labels[i]
        same_label_indices = np.where(gallery_labels == true_label)[0]
        if len(same_label_indices) == 0:
            continue
        _, ranked_indices = torch.sort(sim_scores, descending=True)
        ranked_indices = ranked_indices.cpu().numpy()
        first_correct = np.where(np.isin(ranked_indices, same_label_indices))[0]
        if len(first_correct)==0:
            continue
        first_rank = first_correct[0] + 1
        ap = 0.0
        num_correct = 0
        for j in range(num_gallery):
            if ranked_indices[j] in same_label_indices:
                num_correct += 1
                ap += num_correct / (j+1)
        ap /= len(same_label_indices)
        ap_list.append(ap)
        if first_rank <= 1: rank1 += 1
        if first_rank <= 3: rank3 += 1
        if first_rank <= 5: rank5 += 1
    map_val = np.mean(ap_list) if ap_list else 0.0
    rank1 = rank1/num_query * 100
    rank3 = rank3/num_query * 100
    rank5 = rank5/num_query * 100
    print(f"Epoch {epoch}: mAP: {map_val:.4f}, Rank@1: {rank1:.2f}%, Rank@3: {rank3:.2f}%, Rank@5: {rank5:.2f}%")
    if log_file:
        with open(log_file, 'a') as f:
            f.write(f"Epoch {epoch}: mAP={map_val:.4f}, Rank@1={rank1:.2f}%, Rank@3={rank3:.2f}%, Rank@5={rank5:.2f}%\n")
    return map_val, rank1, rank3, rank5

# Training loop
evaluation_interval = 1
best_map = 0.0

for epoch in range(num_epochs):
    model.train()
    total_loss, total_samples = 0.0, 0
    if (epoch + 1) % evaluation_interval == 0:
        mAP, r1, r3, r5 = evaluate(model, gallery_test_loader, test_loader, target_classes=None, device=device, epoch=epoch+1)
        if mAP > best_map:
            best_map = mAP
            torch.save(model.state_dict(), "best_projection_model.pth")
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for embeddings_batch, labels_batch, _ in pbar:
        embeddings_batch, labels_batch = embeddings_batch.to(device), labels_batch.to(device)
        optimizer.zero_grad()
        outputs = model(embeddings_batch)
        loss = criterion(outputs, labels_batch)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * embeddings_batch.size(0)
        total_samples += embeddings_batch.size(0)
        pbar.set_description(f"Epoch {epoch+1}, Loss: {total_loss/total_samples:.4f}")
    avg_loss = total_loss / total_samples
    print(f"Epoch {epoch+1}, Final Loss: {avg_loss:.4f}")
    # Plot clusters
    query_emb, query_labels, query_is_gallery = get_embeddings(model, test_loader, target_classes=None, device=device, is_gallery=False)
    gallery_emb, gallery_labels, gallery_is_gallery = get_embeddings(model, gallery_test_loader, target_classes=None, device=device, is_gallery=True)
    combined_emb = np.vstack([query_emb, gallery_emb])
    combined_labels = np.hstack([query_labels, gallery_labels])
    combined_is_gallery = np.hstack([query_is_gallery, gallery_is_gallery])
    plot_clusters(combined_emb, combined_labels, target_classes=None, epoch=epoch+1, is_gallery=combined_is_gallery)
